In [ ]:
import os
import struct
from pathlib import Path

from functools import lru_cache


def Tab(n):
    if n < 0:
        return 0
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


def Opt(n):
    if n < 0:
        return 0
    if n <= 1:
        return 1
    count1 = count2 = 1
    for _ in range(2, n + 1):
        count1, count2 = count2, count1 + count2
    return count2


def Memoiz(n):
    if n < 0:
        return 0

    @lru_cache(maxsize=None)
    def ways(step):
        if step <= 1:
            return 1
        return ways(step - 1) + ways(step - 2)

    return ways(n)


def grab(nums):
    if not nums:
        return 0
    if len(nums) == 1:
        return nums[0]

    max2 = nums[0]
    max1 = max(nums[0], nums[1])
    for i in range(2, len(nums)):
        current = max(max1, max2 + nums[i])
        max2, max1 = max1, current
    return max1


def house_robber_memo(nums):
    @lru_cache(maxsize=None)
    def best(index):
        if index < 0:
            return 0
        return max(best(index - 1), nums[index] + best(index - 2))

    return best(len(nums) - 1)


def house_robber_tabulation(nums):
    dp = [0] * (len(nums) + 1)
    for count in range(1, len(nums) + 1):
        skipped = dp[count - 1]
        robbed = nums[count - 1] + (dp[count - 2] if count >= 2 else 0)
        dp[count] = max(skipped, robbed)
    return dp[-1]



def read_idx_images(file_path):
    with open(file_path, "rb") as file:
        magic, count, rows, columns = struct.unpack(">IIII", file.read(16))
        if magic != 2051:
            raise ValueError(f"Неверный IDX-файл изображений: {file_path}")
        images = bytearray(file.read())
    import numpy as np

    return np.frombuffer(images, dtype=np.uint8).reshape(count, rows * columns)


def read_idx_labels(file_path):
    with open(file_path, "rb") as file:
        magic, count = struct.unpack(">II", file.read(8))
        if magic != 2049:
            raise ValueError(f"Неверный IDX-файл меток: {file_path}")
        labels = bytearray(file.read())
    import numpy as np

    return np.frombuffer(labels, dtype=np.uint8).reshape(count)


def find_file(root, *names):
    for name in names:
        for file_path in Path(root).rglob(name):
            if file_path.is_file():
                return file_path
    raise FileNotFoundError(f"Файл {names} не найден в {root}")


def mnist_classification():
    try:
        import kagglehub
        import matplotlib.pyplot as plt
        import numpy as np
        from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
        from sklearn.metrics import accuracy_score
        from sklearn.model_selection import train_test_split
        from sklearn.tree import DecisionTreeClassifier
    except ImportError as error:
        raise SystemExit(
            "Для этого задания установите: pip install kagglehub numpy scikit-learn matplotlib"
        ) from error

    print("Скачиваем MNIST...")
    cache_root = Path.home() / ".cache" / "kagglehub" / "datasets" / "hojjatk" / "mnist-dataset" / "versions" / "1"
    if (cache_root / "train-images.idx3-ubyte").is_file():
        root = cache_root
    else:
        root = Path(kagglehub.dataset_download("hojjatk/mnist-dataset"))
    print("Путь к датасету:", root)

    x_train_all = read_idx_images(
        find_file(root, "train-images-idx3-ubyte", "train-images.idx3-ubyte")
    )
    y_train_all = read_idx_labels(
        find_file(root, "train-labels-idx1-ubyte", "train-labels.idx1-ubyte")
    )
    x_test = read_idx_images(
        find_file(root, "t10k-images-idx3-ubyte", "t10k-images.idx3-ubyte")
    )
    y_test = read_idx_labels(
        find_file(root, "t10k-labels-idx1-ubyte", "t10k-labels.idx1-ubyte")
    )


    x_train, _, y_train, _ = train_test_split(
        x_train_all,
        y_train_all,
        train_size=12000,
        random_state=42,
        stratify=y_train_all,
    )

    print("Размер train:", x_train.shape, "Размер test:", x_test.shape)
    classes, counts = np.unique(y_train, return_counts=True)
    print("Баланс классов в train:")
    for digit, count in zip(classes, counts):
        print(f"Цифра {digit}: {count}")


    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for digit, ax in enumerate(axes.flat):
        index = np.where(y_train_all == digit)[0][0]
        ax.imshow(x_train_all[index].reshape(28, 28), cmap="gray")
        ax.set_title(f"Метка: {digit}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    models = {
        "Decision Tree без ограничения глубины": DecisionTreeClassifier(random_state=42),
        "Decision Tree, max_depth=15": DecisionTreeClassifier(max_depth=15, random_state=42),
        "Random Forest": RandomForestClassifier(
            n_estimators=100, max_depth=20, n_jobs=-1, random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=50, max_depth=2, random_state=42
        ),
    }

    print("\nСравнение моделей по accuracy:")
    results = {}
    for name, model in models.items():
        print("Обучается:", name)
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)
        results[name] = accuracy_score(y_test, predictions)
        print(f"{name}: accuracy = {results[name]:.4f}")

    print("\nЛучшая модель:", max(results, key=results.get))
    print(
        "Дерево без ограничения глубины обычно переобучается; сравните его "
        "результат с деревом ограниченной глубины и ансамблями."
    )


if __name__ == "__main__":
    print(Tab(4))
    print(Opt(4))
    print(Memoiz(6))
    print("House Robber:", grab([1, 2, 3, 1]))
    print("House Robber:", grab([2, 7, 9, 3, 1]))
    mnist_classification()


- Climbing Stairs: `Memoiz` — `O(n)` времени и `O(n)` памяти; `Tab` — `O(n)` времени и `O(n)` памяти; `Opt` — `O(n)` времени и `O(1)` дополнительной памяти.
- House Robber: memoization — `O(n)` времени и `O(n)` памяти; табуляция — `O(n)` и `O(n)`; `grab` — `O(n)` и `O(1)`.
- MNIST обучался на стратифицированной выборке из 12 000 изображений, оценка — на полном test из 10 000 изображений. Accuracy: дерево без ограничения глубины — `0.8318`, дерево с `max_depth=15` — `0.8341`, Random Forest — `0.9534`, Gradient Boosting — `0.8866`. Лучший результат показал Random Forest.
- При `N` обучающих изображениях и `D=784` признаках обучение дерева в среднем занимает `O(ND log N)`; худший случай — `O(N²D)`. Для ансамблей стоимость примерно пропорциональна числу деревьев. Память для набора данных — `O(ND)`, модели — зависит от размера деревьев.